# Hydrologic Zone ODE Solver with PyTorch AD

This notebook demonstrates the Surface hydrologic zone with a custom ordinary differential equation solver
that uses PyTorch automatic differentiation instead of finite-difference or built-in root-finders.

The surface zone follows a water balance equation where storage change depends on precipitation, evaporation,
and lateral drainage. At each timestep, we solve the implicit Euler equation `f(s) = 0` using PyTorch autograd.

Every hot tensor kernel is wrapped in `torch.compile` (via the `compile_gate` below), while the AD Newton
iteration itself stays eager -- see the notes in the `ad_solver` docstring for why this split is required on
torch 2.14.


In [9]:
from __future__ import annotations
from time import time_ns
from dataclasses import dataclass
from collections.abc import Callable
from typing import Any, TypeAlias, TypeVar, cast
import numpy as np
import torch
from torch import Tensor

# ---------------------------------------------------------------------------
# Numerical-precision / device policy (self-contained mirror of potions.device)
# ---------------------------------------------------------------------------
torch.manual_seed(0)
DEVICE: torch.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
# DEVICE: torch.device = torch.device("cpu")
DTYPE: torch.dtype = torch.float32 if DEVICE.type == "cuda" else torch.float64

# Solver hyper-parameters (module-level constants)
MAX_ITER: int = 50
ATOL: float = 1e-8

Function = Callable[[Tensor, Tensor, Tensor, Tensor], Tensor]

compile_enabled: bool = True


def compile_function(fn: Function) -> Function:
    """Wrap ``fn`` in ``torch.compile``

    Applied via ``@compile_gate`` on the pure-tensor residual kernels so the
    compiled wrapper object is built exactly once at class-definition time;
    ``torch.compile`` traces lazily on the first call, so cost is negligible.

    ``torch._dynamo.config.suppress_errors = True`` lets a backend codegen
    failure fall back to eager execution instead of aborting the run.
    """
    torch._dynamo.config.suppress_errors = True
    return cast(Function, torch.compile(fn, dynamic=True))  # type: ignore

## Surface Zone Parameters

The SurfaceZone represents the uppermost soil layer. Key parameters control:
- `fc`: Field capacity (maximum storage)
- `lp`: Lowering potential parameter (drainage nonlinearity)
- `beta`: Baseline evaporation coefficient
- `k0`: Base drainage conductance
- `thr`: Residual moisture threshold


In [10]:
@dataclass
class SurfaceZone:
    """Surface hydrologic zone solving the implicit-Euler water balance with a
    PyTorch automatic-differentiation (AD) Newton root-finder.

    Governing equations
    -------------------
    The surface zone stores water ``s`` [mm] in ``[0, fc]``. Its rate of change
    is the water balance::

        dS/dt = P_eff - E - D

    with
        P_eff = p * (1 - min(s, 0.2 fc) / (0.2 fc))      (interception)
        E     = PET * beta * (s / fc)                     (evaporation)
        D     = k0 * (s / fc) ** lp * [s > thr]           (drainage, gated)

    At each timestep the *implicit* Euler step ``f(s) = 0`` is solved for the
    new storage ``s``::

        f(s) = (s0 - s) + dt * (dS/dt)(s) = 0

    The root is located with a Newton iteration whose derivative ``df/ds`` is
    obtained analytically via ``torch.autograd.grad`` (no finite differences).
    Each update is clamped to the physical range ``[0, fc]`` with ``torch.where``.
    """

    fc: Tensor
    lp: Tensor
    beta: Tensor
    k0: Tensor
    thr: Tensor

    def vert_flux(self, s: Tensor, precip: Tensor, temp: Tensor, pet: Tensor) -> Tensor:
        return precip * (s / self.fc).pow(self.beta)

    def lat_flux(self, s: Tensor, precip: Tensor, temp: Tensor, pet: Tensor) -> Tensor:
        return (self.k0 * (s - self.thr)).clamp_min(0.0)

    def vap_flux(self, s: Tensor, precip: Tensor, temp: Tensor, pet: Tensor) -> Tensor:
        factor = s / (self.fc * self.lp)
        return (factor * pet).clamp_max(1.0)

    def forc_flux(self, s: Tensor, precip: Tensor, temp: Tensor, pet: Tensor) -> Tensor:
        return precip

    def mass_balance(
        self, s: Tensor, precip: Tensor, temp: Tensor, pet: Tensor
    ) -> Tensor:
        """Compute the water-balance rate ``dS/dt`` [mm/day] at storage ``s``.

        Pure tensor arithmetic (no autograd); a leaf of the residual graph and
        therefore safe to ``torch.compile`` on the fake-tensor backend.

        Args:
            s: Current storage [mm] (``requires_grad True``)
            precip: Precipitation rate [mm/day]
            potential_et: Potential evapotranspiration [mm/day]

        Returns:
            ``dS/dt`` [mm/day] as a scalar tensor.
        """
        return (
            self.forc_flux(s, precip, temp, pet)
            - self.vap_flux(s, precip, temp, pet)
            - self.lat_flux(s, precip, temp, pet)
            - self.vert_flux(s, precip, temp, pet)
        )

    def residual(
        self,
        s: Tensor,
        s0: Tensor,
        dt: float,
        precip: Tensor,
        temp: Tensor,
        potential_et: Tensor,
    ) -> Tensor:
        """Implicit-Euler residual ``f(s) = (s0 - s) + dt * dS/dt(s)``.

        The root ``f(s) = 0`` is the target storage; differentiated by autograd
        inside :meth:`ad_solver`. Compiled like :meth:`mass_balance` (no autograd
        lives here).

        Args:
            s: Current storage guess [mm] (``requires_grad True``)
            s0: Previous storage [mm]
            dt: Timestep [days]
            precip: Precipitation [mm/day]
            potential_et: Potential ET [mm/day]

        Returns:
            Residual ``f(s)`` as a scalar tensor.
        """
        residual_value: Tensor = self.mass_balance(s, precip, temp, potential_et)
        return (s0 - s) + dt * residual_value

    def step(
        self,
        s0: Tensor,
        dt: float,
        precip: Tensor,
        temp: Tensor,
        potential_et: Tensor,
        max_iterations: int = 20,
        absolute_tolerance: float = 1e-6,
        relative_tolerance: float = 1e-6,
        minimum_derivative: float = 1e-12,
    ) -> Tensor:
        """Solve the implicit-Euler residual with batched Newton iterations."""

        s = s0.detach().clone()

        for _ in range(max_iterations):
            s = s.detach().requires_grad_(True)

            residual_value = self.residual(
                s,
                s0,
                dt,
                precip,
                temp,
                potential_et,
            )

            derivative = torch.autograd.grad(
                outputs=residual_value,
                inputs=s,
                grad_outputs=torch.ones_like(residual_value),
                create_graph=False,
            )[0]

            safe_derivative = torch.where(
                derivative.abs() >= minimum_derivative,
                derivative,
                torch.where(
                    derivative >= 0,
                    torch.full_like(derivative, minimum_derivative),
                    torch.full_like(derivative, -minimum_derivative),
                ),
            )

            candidate = s - residual_value / safe_derivative

            tolerance = absolute_tolerance + relative_tolerance * s.abs()
            converged = residual_value.abs() <= tolerance

            s = torch.where(converged, s, candidate)

        return s.detach()

## Custom ODE Solver Demonstration

We advance the system with the PyTorch autograd-based solver in `SurfaceZone`.
This avoids `scipy.integrate` and finite-difference methods, instead using autograd to
compute `df/ds` and perform Newton-style updates at each timestep.


In [15]:
n_sets = 500_000


def get_range(min_val: float, max_val: float) -> Tensor:
    x = np.random.rand(n_sets)

    x_scaled = x * (max_val - min_val) + min_val

    return Tensor(x_scaled).to(DEVICE)


# Create the surface zone instance with the AD solver
fcs = get_range(100, 1000)
lps = get_range(0.05, 1.0)
betas = get_range(0.5, 3.0)
k0s = get_range(0.05, 0.5)
thrs = get_range(1, 100)
zone = SurfaceZone(fc=fcs, lp=lps, beta=betas, k0=k0s, thr=thrs)

# Time parameters
dt: float = 1.0  # days
n_steps: int = 3650

zones_solved: int = n_sets * n_steps

# Forcing series (simple sinusoidal pattern)
t: Tensor = Tensor(np.linspace(0, 10, n_steps)).to(DEVICE)
precip: Tensor = 5.0 + 3.0 * torch.sin(t)  # mm/day, ranges 2-8
pet: Tensor = 3.0 + 1.0 * torch.sin(t + 0.5)  # mm/day, ranges 2-4
temp: Tensor = 10.0 + (20.0 * t)
ss: Tensor = 0.5 * fcs

# Run the simulation with the custom solver
start = time_ns()
for i in range(n_steps):
    t_i = t[i]
    ppt_i = precip[i]
    pet_i = pet[i]
    s: Tensor = zone.step(ss, dt, ppt_i, t_i, pet_i)
end = time_ns()
duration = (end - start) / 10**9

speed: float = duration / zones_solved * 1000
rate: float = int(zones_solved / duration)
sim_rate = int(rate / 3_650 / 6)

print(f"Solved {rate:.2e} zones per second")
print(f"Can solve {sim_rate} 3-layer simulations per second")
print(f"Took {speed:.4f} ms per zone")

Solved 6.54e+07 zones per second
Can solve 2986 3-layer simulations per second
Took 0.0000 ms per zone


## Visualization

Plot the storage evolution over the simulation period, showing precipitation and ET forcing.


In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 6))

# Storage
time_axis: npt.NDArray[np.float64] = np.linspace(0, dt * n_steps, len(storage_history))
ax1.plot(time_axis, storage_history.numpy(), "b-", linewidth=2, label="Storage S [mm]")
ax1.set_xlabel("Time [days]")
ax1.set_ylabel("Storage S [mm]", color="b")
ax1.tick_params(axis="y", labelcolor="b")
ax1.set_ylim(0, zone.fc * 1.1)

# Secondary axis for forcing
ax2 = ax1.twinx()
ax2.plot(time_axis[:n_steps], precip, "g--", alpha=0.7, label="Precipitation [mm/day]")
ax2.plot(time_axis[:n_steps], pet, "r--", alpha=0.7, label="Potential ET [mm/day]")
ax2.set_ylabel("Forcing [mm/day]", color="k")
ax2.tick_params(axis="y", labelcolor="k")

plt.title("Surface Hydrologic Zone Simulation with PyTorch AD ODE Solver")
fig.tight_layout()
plt.show()

/tmp/ipykernel_2368609/995951724.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Summary**: This notebook demonstrates the Surface hydrologic zone with a PyTorch automatic differentiation-based ODE solver.
- The zone solves an implicit Euler equation `f(s) = (s0 - s) + dt * mass_balance(s) = 0` at each timestep
- PyTorch autograd computes `df/ds` analytically via `torch.autograd.grad` for Newton-style updates
- The hot tensor kernels (`mass_balance`, `implicit_function`) are wrapped in `torch.compile` via `compile_gate`;
  the AD Newton iteration stays eager because `torch.autograd.grad` is not traceable on torch 2.14's compile backend
- The simulation shows storage dynamics driven by precipitation and evapotranspiration forcing
- Key zone parameters (`fc`, `lp`, `beta`, `k0`, `thr`) control the nonlinear dynamics
- Automatic differentiation provides exact gradients (no finite-difference approximation error)
- The solver uses clamped updates via `torch.where` to keep storage in physical range [0, fc]
